<div align="center">

# YAPAY ZEKÂ DESTEKLİ İLAÇ TASARIMI

**Erzincan Binali Yıldırım Üniversitesi — Sürekli Eğitim Uygulama ve Araştırma Merkezi**

28–29 Kasım 2026 · ML tabanlı QSAR uygulamalı eğitimi

</div>

---

### Eğitim sorumlusu

**Dr. Öğr. Üyesi Ekrem Yaşar**
Erzincan Binali Yıldırım Üniversitesi, Biyofizik Anabilim Dalı

📧 `ey.biophysics@gmail.com`

---

# Oturum 5 — Model Karşılaştırma, Açıklanabilirlik (SHAP) ve Uygulanabilirlik Alanı

> **Girdi:** Oturum 3 ve 4'ün sonuçları ve kaydedilmiş modelleri
> **Çıktı:** karşılaştırma tablosu, SHAP grafikleri, `ntrk1_ad_parametreleri.json`

Elimizde eğitilmiş modeller var ve iyi skorlar alıyorlar. Ama bir QSAR
çalışması burada bitmez. Üç soru daha yanıtlanmalı:

| Soru | Yöntem | Bu oturumdaki bölüm |
|---|---|---|
| Hangi model/temsil en iyisi ve neden? | Tüm sonuçları yan yana koymak | Bölüm 2–3 |
| Model **neden** böyle tahmin ediyor? | **SHAP** (açıklanabilir YZ) | Bölüm 4–6 |
| Model **hangi moleküller için** güvenilir? | **Uygulanabilirlik alanı** (Williams grafiği, Tanimoto) | Bölüm 7–9 |

## Neden bu üçü zorunlu?

**Açıklanabilirlik.** Bir medisinal kimyacıya "model bu molekülün potent
olduğunu söylüyor" demek yetmez. Kimyacı *hangi özellik yüzünden* sorusunu
sorar; çünkü bir sonraki molekülü o bilgiyle tasarlayacaktır. Açıklanamayan
bir model, karar vermekte kullanılamaz.

**Uygulanabilirlik alanı (Applicability Domain, AD).** Bir model yalnızca
**eğitildiği kimyasal uzayda** güvenilirdir. O uzayın dışındaki bir moleküle
de bir sayı üretir — ama o sayı bir tahmin değil, bir **ekstrapolasyondur**.
Hangi tahminlere güvenebileceğimizi bilmeden sanal tarama yapmak, pusulasız
yola çıkmaktır. OECD'nin QSAR model doğrulama ilkelerinde AD tanımı
**zorunlu** bir madde olarak yer alır.

Oturum 6'da yeni moleküller için tahmin üretirken bu oturumda belirlediğimiz
AD sınırlarını kullanacağız.

---
## 1. Yapılandırma ve kurulum

In [ ]:
TARGET_ADI  = "NTRK1"
HEDEF_KOLON = "pIC50"
ESIK        = 7.5

DRIVE_PROJE_KLASORU = "/content/drive/MyDrive/EBYUSEM2026_MLQSAR"
YEREL_KLASOR        = "EBYUSEM2026_MLQSAR"

# Oturum 3-4 ciktilari
AD_REG_SONUC = "ntrk1_regresyon_sonuclari.csv"
AD_SNF_SONUC = "ntrk1_siniflandirma_sonuclari.csv"
AD_REG_MODEL = "ntrk1_en_iyi_regresyon_modeli.joblib"
AD_SNF_MODEL = "ntrk1_en_iyi_siniflandirma_modeli.joblib"
AD_DESC      = "ntrk1_descriptors.csv"
AD_ECFP      = "ntrk1_ecfp4.csv"

# SHAP ayarlari
SHAP_ORNEK     = 600    # SHAP hesabi icin kullanilacak molekul sayisi (hiz icin)
SHAP_ILK_N     = 15     # grafiklerde gosterilecek ozellik sayisi

# Uygulanabilirlik alani: Tanimoto esigi (gerekcesi Bolum 9'da)
TANIMOTO_ESIGI = 0.30   # ECFP4 ile "ilgili kimyasal sinif" sinirinda yaygin deger

RASTGELE_TOHUM = 42

print(f"Hedef: {TARGET_ADI}")

In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJE = Path(DRIVE_PROJE_KLASORU)
    ORTAM = "Colab + Drive"
except (ImportError, ModuleNotFoundError):
    PROJE = Path(YEREL_KLASOR)
    ORTAM = "Yerel"

PROJE.mkdir(parents=True, exist_ok=True)
print(f"Ortam         : {ORTAM}")
print(f"Proje klasoru : {PROJE}")

In [ ]:
!pip install shap rdkit -q
print("Kurulum tamam.")

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import json

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import Draw, rdFingerprintGenerator
RDLogger.DisableLog("rdApp.*")

plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", 40)

print(f"shap {shap.__version__} hazir.")

---
## 2. Bütün sonuçları bir araya getirmek

Oturum 3 ve 4'ün kaydettiği sonuç tablolarını okuyoruz. Regresyon ve
sınıflandırma farklı metrikler kullandığı için ikisini ayrı ayrı gösterip
sonra ortak bir özet çıkarıyoruz.

In [ ]:
def oku(ad):
    yol = PROJE / ad
    if yol.exists():
        return pd.read_csv(yol)
    print(f"  !! {ad} bulunamadi — ilgili oturumu calistirdiniz mi?")
    return None


df_reg = oku(AD_REG_SONUC)
df_snf = oku(AD_SNF_SONUC)

tarama_reg = oku("ntrk1_regresyon_lazypredict_tarama.csv")
tarama_snf = oku("ntrk1_siniflandirma_lazypredict_tarama.csv")

if df_reg is None and df_snf is None:
    raise FileNotFoundError(
        "Hic sonuc dosyasi bulunamadi.\n"
        "Once Oturum 3 ve Oturum 4'u ayni Drive klasorunu kullanarak "
        "calistirin."
    )

print("Yuklenen sonuc dosyalari:")
for ad, d in [("regresyon", df_reg), ("siniflandirma", df_snf)]:
    if d is not None:
        print(f"  {ad:<16} {d.shape[0]} satir")

In [ ]:
if df_reg is not None:
    print("REGRESYON SONUCLARI")
    print("=" * 100)
    kolonlar = [k for k in ["Model", "Ozellik seti", "Bolunme", "CV R2 (Q2)",
                            "Egitim R2", "Test R2", "Test RMSE", "Test MAE",
                            "y-random CV R2 (max)"] if k in df_reg.columns]
    print(df_reg[kolonlar].to_string(index=False))

In [ ]:
if df_snf is not None:
    print("SINIFLANDIRMA SONUCLARI")
    print("=" * 110)
    kolonlar = [k for k in ["Model", "Ozellik seti", "Karar esigi",
                            "Temel cizgi dogruluk", "Test Dogruluk",
                            "Test Dengeli dogruluk", "Test MCC",
                            "Test ROC-AUC", "Test PR-AUC",
                            "y-random CV ROC-AUC (max)"]
                if k in df_snf.columns]
    print(df_snf[kolonlar].to_string(index=False))

### LazyPredict taramalarının özeti

Oturum 3 ve 4'te onlarca modeli taramıştık. O tabloları burada birleştirip
iki soruya bakıyoruz:

1. **Hangi temsil** (betimleyici / MACCS / ECFP4) daha iyi sonuç veriyor?
2. **Hangi model aileleri** sürekli üstte çıkıyor?

İkinci soru önemlidir: tek bir modelin bir kez iyi skor alması şans olabilir,
ama bir **ailenin** (ör. ağaç toplulukları) sürekli üstte olması gerçek bir
örüntüdür.

In [ ]:
fig, eksenler = plt.subplots(1, 2, figsize=(15, 4.6))

for eksen, (tarama, metrik, baslik) in zip(eksenler, [
    (tarama_reg, "R-Squared", "Regresyon — LazyPredict"),
    (tarama_snf, "Balanced Accuracy", "Siniflandirma — LazyPredict"),
]):
    if tarama is None or metrik not in tarama.columns:
        eksen.text(0.5, 0.5, "Tarama verisi yok", ha="center", va="center",
                   transform=eksen.transAxes)
        eksen.axis("off")
        continue
    for etiket, grup in tarama.groupby("Ozellik seti"):
        ilk = grup.sort_values(metrik, ascending=False).head(10)
        eksen.plot(range(1, len(ilk) + 1), ilk[metrik], marker="o",
                   label=f"{etiket} (en iyi {ilk[metrik].iloc[0]:.3f})")
    eksen.set_xlabel("Siralama (en iyi -> 10.)")
    eksen.set_ylabel(metrik)
    eksen.set_title(baslik)
    eksen.legend(fontsize=8)
    eksen.grid(alpha=0.3, linestyle="--")

plt.tight_layout()
plt.savefig(PROJE / "ntrk1_lazypredict_karsilastirma.png", dpi=150,
            bbox_inches="tight")
plt.show()

In [ ]:
# En iyi 10 modelde hangi aileler one cikiyor?
def aile(model_adi) -> str:
    ad = str(model_adi).lower()
    if any(x in ad for x in ["forest", "extratree", "bagging"]):
        return "Agac toplulugu (bagging)"
    if any(x in ad for x in ["xgb", "lgbm", "gradientboost", "hist", "adaboost",
                             "catboost"]):
        return "Gradyan artirma (boosting)"
    if any(x in ad for x in ["svr", "svc", "nusv"]):
        return "Destek vektor makinesi"
    if "neighbor" in ad or "knn" in ad:
        return "Komsuluk (kNN)"
    if any(x in ad for x in ["ridge", "lasso", "elastic", "linear", "lars",
                             "logistic", "sgd", "bayesian", "huber", "passive",
                             "perceptron", "ransac", "theil", "ard", "orthogonal",
                             "tweedie", "poisson", "gamma", "quantile"]):
        return "Lineer model"
    if "tree" in ad:
        return "Tek karar agaci"
    if "gaussian" in ad:
        return "Gauss sureci"
    return "Diger"


for tarama, metrik, baslik in [(tarama_reg, "R-Squared", "REGRESYON"),
                               (tarama_snf, "Balanced Accuracy", "SINIFLANDIRMA")]:
    if tarama is None or metrik not in tarama.columns:
        continue
    ilk10 = (tarama.sort_values(metrik, ascending=False)
             .groupby("Ozellik seti").head(10).copy())
    # Model adi, CSV'den okundugunda "Model" kolonunda; dogrudan
    # LazyPredict'ten geldiginde ise indekste bulunur.
    if "Model" in ilk10.columns:
        model_adlari = ilk10["Model"].astype(str)
    else:
        model_adlari = pd.Series(ilk10.index.astype(str), index=ilk10.index)
    ilk10["Aile"] = [aile(a) for a in model_adlari]
    print(f"\n{baslik} — en iyi 10'da aile dagilimi")
    print("-" * 64)
    ozet = (ilk10.groupby("Aile")[metrik]
            .agg(["count", "max", "mean"])
            .rename(columns={"count": "Kac kez ilk 10'da",
                             "max": f"En iyi {metrik}",
                             "mean": f"Ortalama {metrik}"})
            .sort_values("Kac kez ilk 10'da", ascending=False))
    print(ozet.round(4).to_string())

---
## 3. Hangi modeli seçmeliyiz?

Karşılaştırma tablosuna bakarken yalnızca en yüksek skora bakmak yetersizdir.
Bir QSAR modelini seçerken şu ölçütleri **birlikte** değerlendirin:

| Ölçüt | Niçin önemli? |
|---|---|
| **Q² (CV R²) / CV ROC-AUC** | Görülmemiş veride beklenen performans. Tek bir test skorundan daha kararlıdır. |
| **Eğitim − CV farkı** | Büyükse aşırı öğrenme var. |
| **y-randomizasyon uçurumu** | Küçükse model şans korelasyonu öğrenmiş olabilir. |
| **İskelet bazlı performans** | Yeni kimyasal serilere genelleme kapasitesi. |
| **Yorumlanabilirlik** | Medisinal kimyacıya ne söyleyebiliyorsunuz? |
| **Karmaşıklık ve hız** | Milyonlarca molekül taranacaksa hız önemlidir. |

> **Pratik tavsiye.** İki model arasında performans farkı küçükse (ör. ΔR² < 0.02),
> **daha basit ve daha yorumlanabilir** olanı seçin. Betimleyici tabanlı bir
> model, 2048 bitlik parmak izi modeline göre biraz düşük skor alsa bile SHAP
> analizi anlamlı olduğu için pratikte daha kullanışlıdır.

In [ ]:
# Kaydedilmis modelleri yukle
paket_reg = paket_snf = None
if (PROJE / AD_REG_MODEL).exists():
    paket_reg = joblib.load(PROJE / AD_REG_MODEL)
    print(f"Regresyon modeli     : {paket_reg['model_adi']}  "
          f"({paket_reg['ozellik_seti']}, {len(paket_reg['ozellik_adlari'])} ozellik)")
    print(f"  CV R2={paket_reg['cv_r2']:.4f}  Test R2={paket_reg['test_r2']:.4f}  "
          f"Test RMSE={paket_reg['test_rmse']:.4f}")
if (PROJE / AD_SNF_MODEL).exists():
    paket_snf = joblib.load(PROJE / AD_SNF_MODEL)
    print(f"\nSiniflandirma modeli : {paket_snf['model_adi']}  "
          f"({paket_snf['ozellik_seti']}, {len(paket_snf['ozellik_adlari'])} ozellik)")
    print(f"  CV ROC-AUC={paket_snf['cv_roc_auc']:.4f}  "
          f"Test ROC-AUC={paket_snf['test_roc_auc']:.4f}  "
          f"Test MCC={paket_snf['test_mcc']:.4f}")

if paket_reg is None:
    raise FileNotFoundError(
        f"'{AD_REG_MODEL}' bulunamadi. Oturum 3'u calistirmaniz gerekiyor."
    )

---
## 4. Özellik matrisini yeniden kurmak

SHAP ve uygulanabilirlik alanı analizleri için modelin eğitildiği özellik
matrisine ve eğitim/test bölünmesine ihtiyacımız var. Bunları Oturum 2 ve 3'ün
kaydettiği dosyalardan geri yüklüyoruz.

In [ ]:
dosya_esleme = {"descriptors": AD_DESC, "ecfp4": AD_ECFP}
ozellik_dosyasi = dosya_esleme.get(paket_reg["ozellik_seti"], AD_DESC)

tablo = pd.read_csv(PROJE / ozellik_dosyasi)
ozellik_adlari = paket_reg["ozellik_adlari"]
X = tablo[ozellik_adlari]
y = tablo[HEDEF_KOLON]
smiles = tablo["Smiles"]

egitim_idx = np.array(paket_reg["egitim_indeksleri"])
test_idx   = np.array(paket_reg["test_indeksleri"])

X_egitim, y_egitim = X.iloc[egitim_idx], y.iloc[egitim_idx]
X_test,   y_test   = X.iloc[test_idx],  y.iloc[test_idx]

boru = paket_reg["boru"]
tahmin_test = boru.predict(X_test)

print(f"Ozellik seti : {paket_reg['ozellik_seti']}  ({X.shape[1]} ozellik)")
print(f"Egitim       : {X_egitim.shape}")
print(f"Test         : {X_test.shape}")
print(f"Model        : {paket_reg['model_adi']}")
print(f"Boru adimlari: {[ad for ad, _ in boru.steps]}")

---
## 5. SHAP: modelin kararını açıklamak

**SHAP** (SHapley Additive exPlanations), oyun teorisindeki **Shapley
değerlerine** dayanır. Temel fikir şudur:

> Bir tahmini, modelin ortalama çıktısından başlayıp her özelliğin katkısını
> tek tek ekleyerek açıklayabiliriz.

Yani her molekül ve her özellik için şunu söyler:

```
tahmin = ortalama_tahmin + katki(MolLogP) + katki(TPSA) + katki(...) + ...
```

Bu **toplamsal** yapı SHAP'ı özel kılar: katkılar tam olarak tahmine toplanır.
Bir özelliğin SHAP değeri pozitifse o özellik tahmini **yukarı**, negatifse
**aşağı** çekmiştir.

**İki düzeyde okunabilir:**

- **Küresel (global):** Bütün veri setinde hangi özellikler önemli?
  → `MolLogP` genel olarak potensi artırıyor mu?
- **Yerel (local):** *Bu belirli molekül* için model neden yüksek tahmin yaptı?
  → Medisinal kimyacı için en değerli kısım budur.

> ⚠️ **Kritik uyarı: SHAP nedensellik değildir.** SHAP, modelin **neye
> baktığını** söyler; doğanın nasıl çalıştığını söylemez. Model yanlış bir
> örüntü öğrenmişse SHAP o yanlış örüntüyü sadakatle gösterir. Ayrıca
> korelasyonlu özellikler katkıyı paylaşır — bu yüzden Oturum 2'de yüksek
> korelasyonlu betimleyicileri elemek SHAP yorumunu anlamlı kılmak için de
> gerekliydi.

In [ ]:
# SHAP hesabi: boru icindeki modeli ve donusturulmus veriyi ayirmamiz gerekir
model = boru.named_steps["model"]
on_islem = boru[:-1] if len(boru.steps) > 1 else None


def donustur(veri):
    if on_islem is None:
        return pd.DataFrame(veri.values, columns=ozellik_adlari,
                            index=veri.index)
    donusmus = on_islem.transform(veri)
    return pd.DataFrame(donusmus, columns=ozellik_adlari, index=veri.index)


# Hiz icin alt ornek
rastgele = np.random.RandomState(RASTGELE_TOHUM)
n_ornek = min(SHAP_ORNEK, len(X_test))
ornek_konum = rastgele.choice(len(X_test), n_ornek, replace=False)
X_shap = donustur(X_test.iloc[ornek_konum])

print(f"SHAP hesaplaniyor ({n_ornek} molekul, {X_shap.shape[1]} ozellik)...")

try:
    # Agac tabanli modeller icin hizli ve kesin yontem
    aciklayici = shap.TreeExplainer(model)
    aciklama = aciklayici(X_shap, check_additivity=False)
    yontem = "TreeExplainer (kesin)"
except Exception as hata:
    print(f"  TreeExplainer kullanilamadi ({type(hata).__name__}); "
          f"model-bagimsiz yonteme geciliyor...")
    arka_plan = shap.utils.sample(donustur(X_egitim), 100,
                                  random_state=RASTGELE_TOHUM)
    aciklayici = shap.Explainer(model.predict, arka_plan)
    aciklama = aciklayici(X_shap)
    yontem = "Permutation/Kernel (yaklasik)"

print(f"Yontem: {yontem}")
print(f"SHAP degerleri sekli: {np.array(aciklama.values).shape}")

# Toplamsallik kontrolu: SHAP katkilari gercekten tahmine topluyor mu?
shap_toplam = np.array(aciklama.values).sum(axis=1) + np.array(aciklama.base_values).ravel()
gercek_tahmin = model.predict(X_shap)
print(f"\nToplamsallik kontrolu — ortalama |fark|: "
      f"{np.abs(shap_toplam - gercek_tahmin).mean():.6f}")
print("(Sifira yakin olmasi, SHAP katkilarinin tahmini tam olarak")
print(" aciklamasi anlamina gelir.)")

### Küresel önem: hangi özellikler modele yön veriyor?

İki grafik çiziyoruz:

- **Çubuk grafiği** — her özelliğin ortalama `|SHAP|` değeri, yani "bu özellik
  tahminleri ortalamada ne kadar oynatıyor?" Sadece **büyüklüğü** gösterir.
- **Arı kümesi (beeswarm)** — her nokta bir molekül. Yatay eksen SHAP katkısı,
  renk ise özelliğin **değerini** gösterir (kırmızı = yüksek, mavi = düşük).
  Bu grafik **yönü** de verir: "bu özellik yüksek olduğunda tahmin artıyor mu?"

Beeswarm'ı okuma kılavuzu: bir satırda kırmızı noktalar sağda, mavi noktalar
solda toplanmışsa, o özellik **arttıkça tahmin artıyor** demektir. Tersi ise
ters ilişkidir. Renkler karışıksa ilişki doğrusal değildir (ya da başka bir
özellikle etkileşim vardır).

In [ ]:
fig = plt.figure(figsize=(9, 6))
shap.plots.bar(aciklama, max_display=SHAP_ILK_N, show=False)
plt.title(f"SHAP kuresel onem — {paket_reg['model_adi']} "
          f"({HEDEF_KOLON} tahmini)", fontsize=11)
plt.tight_layout()
plt.savefig(PROJE / "ntrk1_shap_onem_cubuk.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig = plt.figure(figsize=(9, 7))
shap.plots.beeswarm(aciklama, max_display=SHAP_ILK_N, show=False)
plt.title("SHAP arı kümesi — özellik değeri ile katkının yönü", fontsize=11)
plt.tight_layout()
plt.savefig(PROJE / "ntrk1_shap_beeswarm.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# En onemli ozellikleri tablo olarak da verelim
ortalama_mutlak = np.abs(np.array(aciklama.values)).mean(axis=0)
onem = (pd.DataFrame({"Ozellik": ozellik_adlari,
                      "Ortalama |SHAP|": ortalama_mutlak})
        .sort_values("Ortalama |SHAP|", ascending=False)
        .reset_index(drop=True))

# Ozellik ile hedef arasindaki yon (Pearson)
onem["Hedefle korelasyon"] = [
    round(float(np.corrcoef(X[o], y)[0, 1]), 3) for o in onem["Ozellik"]
]
onem["Yon"] = np.where(onem["Hedefle korelasyon"] > 0, "artirir", "azaltir")

print(f"En onemli {SHAP_ILK_N} ozellik")
print("=" * 70)
print(onem.head(SHAP_ILK_N).round(4).to_string(index=False))

onem.to_csv(PROJE / "ntrk1_shap_ozellik_onemi.csv", index=False)
print(f"\nKaydedildi: ntrk1_shap_ozellik_onemi.csv")

### Bağımlılık grafiği: bir özelliğin etkisi nasıl değişiyor?

Bağımlılık (dependence) grafiği, tek bir özelliğin **değeri** ile onun **SHAP
katkısı** arasındaki ilişkiyi gösterir. Doğrusal bir ilişki mi, eşikli mi,
yoksa optimumu olan bir eğri mi?

İlaç tasarımında bu çok işe yarar: örneğin lipofilikliğin (`MolLogP`) belirli
bir aralıkta potensi artırıp sonra düşürdüğünü görürseniz, bu doğrudan bir
tasarım kuralıdır.

In [ ]:
en_onemli = onem["Ozellik"].head(4).tolist()

fig, eksenler = plt.subplots(2, 2, figsize=(12.5, 8))
for eksen, ozellik in zip(eksenler.ravel(), en_onemli):
    konum = ozellik_adlari.index(ozellik)
    eksen.scatter(np.array(aciklama.data)[:, konum],
                  np.array(aciklama.values)[:, konum],
                  s=12, alpha=0.5, color="#4878a8")
    eksen.axhline(0, color="k", linestyle="--", linewidth=1)
    eksen.set_xlabel(f"{ozellik} (olcekli deger)" if on_islem is not None
                     else ozellik)
    eksen.set_ylabel(f"SHAP katkisi ({HEDEF_KOLON})")
    eksen.set_title(ozellik, fontsize=10)
    eksen.grid(alpha=0.3, linestyle="--")

plt.suptitle("SHAP bagimlilik grafikleri — en onemli dort ozellik", y=1.0)
plt.tight_layout()
plt.savefig(PROJE / "ntrk1_shap_bagimlilik.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 6. Yerel açıklama: *bu* molekül neden potent?

Şimdi tek bir moleküle odaklanıyoruz. **Şelale (waterfall) grafiği**, modelin
ortalama tahmininden başlayıp her özelliğin katkısını ekleyerek nihai tahmine
nasıl ulaştığını gösterir.

İki molekül seçiyoruz: modelin **en potent** ve **en zayıf** gördüğü test
molekülleri. Yapılarını da çizip SHAP açıklamasıyla birlikte okuyacağız.

Bu, medisinal kimyacıyla konuşmanın dilidir: "model bu molekülü potent buluyor
çünkü X özelliği yüksek ve Y özelliği düşük."

In [ ]:
# Alt ornek icindeki en yuksek ve en dusuk tahminli molekuller
tahminler_ornek = model.predict(X_shap)
en_yuksek = int(np.argmax(tahminler_ornek))
en_dusuk  = int(np.argmin(tahminler_ornek))

for konum, etiket in [(en_yuksek, "EN POTENT (model tahminine gore)"),
                      (en_dusuk, "EN ZAYIF (model tahminine gore)")]:
    kuresel_konum = test_idx[ornek_konum[konum]]
    print(f"\n{'=' * 72}")
    print(f"{etiket}")
    print(f"  Tahmin  {HEDEF_KOLON} = {tahminler_ornek[konum]:.2f}")
    print(f"  Gercek  {HEDEF_KOLON} = {y.iloc[kuresel_konum]:.2f}")
    print(f"  SMILES: {smiles.iloc[kuresel_konum]}")

    fig = plt.figure(figsize=(9, 5.5))
    shap.plots.waterfall(aciklama[konum], max_display=12, show=False)
    plt.title(etiket, fontsize=10)
    plt.tight_layout()
    plt.show()

In [ ]:
# Bu iki molekulun yapilari
moller, etiketler = [], []
for konum, etiket in [(en_yuksek, "en potent"), (en_dusuk, "en zayif")]:
    kuresel_konum = test_idx[ornek_konum[konum]]
    mol = Chem.MolFromSmiles(smiles.iloc[kuresel_konum])
    if mol is not None:
        moller.append(mol)
        etiketler.append(f"{etiket}\ntahmin {tahminler_ornek[konum]:.2f} / "
                         f"gercek {y.iloc[kuresel_konum]:.2f}")

Draw.MolsToGridImage(moller, legends=etiketler, molsPerRow=2,
                     subImgSize=(380, 300))

---
## 7. Uygulanabilirlik alanı: Williams grafiği

Şimdi kursun en pratik kavramına geliyoruz.

### Kaldıraç (leverage) nedir?

Bir molekülün **kaldıracı** (`h`), onun özellik uzayında eğitim verisinin
merkezinden ne kadar uzak olduğunu ölçer. Şöyle hesaplanır:

$$h_i = \mathbf{x}_i^T (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{x}_i$$

Yüksek kaldıraç = molekül, eğitim verisinin kimyasal uzayının **kenarında ya
da dışında**. Model onun için tahmin üretir, ama bu bir **ekstrapolasyondur**.

### Kritik kaldıraç `h*`

Eşik olarak şu kullanılır:

$$h^* = \frac{3(p+1)}{n}$$

`p` = özellik sayısı, `n` = eğitim molekülü sayısı. `h > h*` olan moleküller
"kaldıraç aykırısı" sayılır.

### Williams grafiği

Yatay eksende kaldıraç, düşey eksende **standartlaştırılmış artıklar**
(standardized residuals). İki eşik çizilir: `h*` (düşey) ve `±3σ` (yatay).
Grafik dört bölgeye ayrılır:

| Bölge | Anlamı | Ne yapmalı? |
|---|---|---|
| `h < h*`, `|artık| < 3` | **Uygulanabilirlik alanı içinde** | Tahmine güvenilebilir |
| `h > h*`, `|artık| < 3` | Kaldıraç aykırısı ama iyi tahmin edilmiş | Dikkatli ol; model şanslı olabilir |
| `h < h*`, `|artık| > 3` | **Yanıt aykırısı** | Veri hatası ya da farklı bağlanma modu olabilir |
| `h > h*`, `|artık| > 3` | Hem uzak hem kötü tahmin | Tahmini kullanma |

> **Niçin bu kadar önemli?** Oturum 6'da binlerce yeni molekülü tarayacağız.
> AD kontrolü olmadan, modelin hiç görmediği kimyasal sınıflardan gelen
> moleküller için ürettiği sayılara güvenmiş oluruz. AD, "bilmiyorum"
> diyebilmenin matematiksel yoludur.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Kaldirac hesabi standartlastirilmis ozellikler uzerinde yapilir.
# Olcekleyici YALNIZCA egitim verisine uydurulur.
olcekleyici_ad = StandardScaler().fit(X_egitim.values)
Xe = olcekleyici_ad.transform(X_egitim.values)
Xt = olcekleyici_ad.transform(X_test.values)

n_egitim, p_ozellik = Xe.shape

# (X^T X)^-1 — tekil olabilecegi icin psodo-ters kullaniyoruz
XtX_ters = np.linalg.pinv(Xe.T @ Xe)


def kaldirac(matris):
    """Her satir icin h = x^T (X^T X)^-1 x"""
    return np.einsum("ij,jk,ik->i", matris, XtX_ters, matris)


h_egitim = kaldirac(Xe)
h_test   = kaldirac(Xt)
h_kritik = 3.0 * (p_ozellik + 1) / n_egitim

# Standartlastirilmis artiklar (egitim artiklarinin std'si ile)
tahmin_egitim = boru.predict(X_egitim)
artik_egitim = y_egitim.values - tahmin_egitim
sigma = artik_egitim.std(ddof=1)

std_artik_egitim = artik_egitim / sigma
std_artik_test   = (y_test.values - tahmin_test) / sigma

print("Uygulanabilirlik alani parametreleri")
print("-" * 58)
print(f"  Egitim molekulu (n)      : {n_egitim}")
print(f"  Ozellik sayisi (p)       : {p_ozellik}")
print(f"  Kritik kaldirac h*       : {h_kritik:.4f}   [3(p+1)/n]")
print(f"  Artik std sapmasi (sigma): {sigma:.4f}")
print(f"\n  Egitim kaldirac araligi  : {h_egitim.min():.4f} - {h_egitim.max():.4f}")
print(f"  Test kaldirac araligi    : {h_test.min():.4f} - {h_test.max():.4f}")

if h_kritik >= 1.0:
    print("\n  !! h* >= 1: ozellik sayisi egitim molekulu sayisina gore yuksek.")
    print("     Kaldirac tabanli AD bu durumda zayif ayirim yapar; Bolum 9'daki")
    print("     Tanimoto benzerligi tabanli AD daha guvenilirdir.")

In [ ]:
fig, eksen = plt.subplots(figsize=(11, 6))

eksen.scatter(h_egitim, std_artik_egitim, s=10, alpha=0.3, color="#95a5a6",
              label=f"Egitim (n={n_egitim})")
eksen.scatter(h_test, std_artik_test, s=18, alpha=0.65, color="#c0392b",
              label=f"Test (n={len(h_test)})")

eksen.axvline(h_kritik, color="#2c3e50", linestyle="--", linewidth=1.8,
              label=f"h* = {h_kritik:.3f}")
eksen.axhline(3, color="#2c3e50", linestyle=":", linewidth=1.8,
              label="+/- 3 sigma")
eksen.axhline(-3, color="#2c3e50", linestyle=":", linewidth=1.8)
eksen.axhline(0, color="k", linewidth=0.8)

eksen.set_xlabel("Kaldirac (leverage, h)")
eksen.set_ylabel("Standartlastirilmis artik")
eksen.set_title(f"Williams grafigi — {paket_reg['model_adi']} "
                f"({TARGET_ADI} {HEDEF_KOLON})")
eksen.legend(fontsize=9, loc="upper right")
eksen.grid(alpha=0.25, linestyle="--")

# Bolge etiketleri
x_ust = eksen.get_xlim()[1]
eksen.text(h_kritik * 0.35, 3.6, "yanit aykirisi", fontsize=8.5, color="#7f8c8d")
eksen.text(h_kritik * 1.08, 3.6, "hem uzak hem kotu", fontsize=8.5, color="#7f8c8d")
eksen.text(h_kritik * 0.35, -0.6, "UYGULANABILIRLIK ALANI", fontsize=9,
           color="#27ae60", weight="bold")
eksen.text(h_kritik * 1.08, -0.6, "kaldirac aykirisi", fontsize=8.5, color="#7f8c8d")

plt.tight_layout()
plt.savefig(PROJE / "ntrk1_williams_grafigi.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Test molekullerini AD bolgelerine ayir
def ad_bolgesi(h, std_artik):
    uzak = h > h_kritik
    kotu = abs(std_artik) > 3
    if not uzak and not kotu:
        return "AD icinde"
    if uzak and not kotu:
        return "Kaldirac aykirisi"
    if not uzak and kotu:
        return "Yanit aykirisi"
    return "Hem uzak hem kotu"


bolgeler = [ad_bolgesi(h, r) for h, r in zip(h_test, std_artik_test)]
df_ad = pd.DataFrame({
    "Smiles": smiles.iloc[test_idx].values,
    "gercek": y_test.values,
    "tahmin": tahmin_test,
    "artik": y_test.values - tahmin_test,
    "kaldirac": h_test,
    "std_artik": std_artik_test,
    "AD_bolgesi": bolgeler,
})

print("Test setinin AD dagilimi")
print("-" * 70)
ozet_ad = (df_ad.groupby("AD_bolgesi")
           .agg(Molekul=("artik", "size"),
                Ortalama_mutlak_hata=("artik", lambda s: s.abs().mean()),
                RMSE=("artik", lambda s: float(np.sqrt((s ** 2).mean()))))
           .sort_values("Molekul", ascending=False))
ozet_ad["Yuzde"] = (100 * ozet_ad["Molekul"] / len(df_ad)).round(1)
print(ozet_ad.round(4).to_string())

df_ad.to_csv(PROJE / "ntrk1_ad_test_analizi.csv", index=False)
print(f"\nKaydedildi: ntrk1_ad_test_analizi.csv")

### AD gerçekten işe yarıyor mu? Asıl sınav

Yukarıdaki tablo bir şey söylüyor olabilir ama onu açıkça sınamak gerekir:

> **Uygulanabilirlik alanı içindeki moleküller, dışındakilerden gerçekten daha
> iyi tahmin ediliyor mu?**

Eğer cevap evetse, AD kullanışlı bir güven ölçüsüdür ve Oturum 6'da ona
dayanabiliriz. Eğer hayırsa, kaldıraç bu veri için iyi bir ölçüt değildir ve
alternatife (Tanimoto benzerliği) geçmemiz gerekir.

Aşağıda kaldıracı beş dilime bölüp her dilimde tahmin hatasını ölçüyoruz.
**Beklenti:** kaldıraç arttıkça hata da artmalı.

In [ ]:
df_ad["kaldirac_dilim"] = pd.qcut(df_ad["kaldirac"], 5,
                                  labels=["1 (en yakin)", "2", "3", "4",
                                          "5 (en uzak)"])

dilim_ozet = (df_ad.groupby("kaldirac_dilim", observed=True)
              .agg(Molekul=("artik", "size"),
                   Ortalama_kaldirac=("kaldirac", "mean"),
                   MAE=("artik", lambda s: s.abs().mean()),
                   RMSE=("artik", lambda s: float(np.sqrt((s ** 2).mean())))))

print("Kaldirac dilimlerine gore tahmin hatasi")
print("=" * 72)
print(dilim_ozet.round(4).to_string())

fig, (sol, sag) = plt.subplots(1, 2, figsize=(13, 4.4))

sol.bar(range(len(dilim_ozet)), dilim_ozet["RMSE"], color="#4878a8",
        edgecolor="white")
sol.set_xticks(range(len(dilim_ozet)))
sol.set_xticklabels(dilim_ozet.index, rotation=20, fontsize=8)
sol.set_ylabel(f"RMSE ({HEDEF_KOLON})")
sol.set_xlabel("Kaldirac dilimi")
sol.set_title("Kaldirac arttikca hata artiyor mu?")
sol.grid(alpha=0.3, axis="y", linestyle="--")

sol_deger = dilim_ozet["RMSE"].iloc[0]
sag_deger = dilim_ozet["RMSE"].iloc[-1]

sag.scatter(df_ad["kaldirac"], df_ad["artik"].abs(), s=14, alpha=0.5,
            color="#c0392b")
sag.axvline(h_kritik, color="#2c3e50", linestyle="--", linewidth=1.6,
            label=f"h* = {h_kritik:.3f}")
sag.set_xlabel("Kaldirac (h)")
sag.set_ylabel(f"|hata| ({HEDEF_KOLON})")
sag.set_title("Kaldirac ile mutlak hata")
sag.legend(fontsize=9)
sag.grid(alpha=0.3, linestyle="--")

plt.tight_layout()
plt.savefig(PROJE / "ntrk1_ad_hata_iliskisi.png", dpi=150, bbox_inches="tight")
plt.show()

korelasyon = df_ad["kaldirac"].corr(df_ad["artik"].abs())
print(f"\nKaldirac ile |hata| arasindaki korelasyon: r = {korelasyon:.3f}")
print(f"En yakin dilim RMSE : {sol_deger:.4f}")
print(f"En uzak dilim RMSE  : {sag_deger:.4f}")
if sag_deger > sol_deger:
    print(f"-> Uzak molekullerde hata %{100 * (sag_deger / sol_deger - 1):.0f} daha yuksek.")
    print("   Kaldirac tabanli AD bu veri icin anlamli bir guven olcusu.")
else:
    print("-> Beklenen egilim gorulmedi. Bu veri icin kaldirac zayif bir olcut;")
    print("   Bolum 9'daki Tanimoto benzerligi tabanli AD'yi tercih edin.")

---
## 8. Alternatif AD: Tanimoto benzerliği

Kaldıraç, özellik uzayındaki uzaklığı ölçer. Kemoinformatikte daha sezgisel
bir alternatif vardır: **yapısal benzerlik**.

Fikir basittir:

> Yeni bir molekülün eğitim setindeki **en benzer** molekülle olan Tanimoto
> benzerliğine bak. Çok düşükse, model bu molekül gibi bir şey hiç görmemiş.

**Tanimoto katsayısı** iki parmak izi arasındaki örtüşmeyi ölçer: 1 = birebir
aynı yapı, 0 = hiç ortak alt-yapı yok. Kemoinformatikte 0.85 üzeri "çok
benzer", 0.3 altı "ilgisiz" kabul edilir (eşikler bağlama göre değişir).

Bu yaklaşımın kaldıraca göre iki avantajı var:

1. **Yorumlanabilir.** "En benzer eğitim molekülü %28 benzer" demek, bir
   kimyacı için "kaldıracı 0.19" demekten çok daha anlamlıdır.
2. **Özellik sayısından bağımsız.** `h*` formülü özellik sayısına duyarlıdır;
   Tanimoto değildir.

Aşağıda her test molekülü için eğitim setine **en yüksek** benzerliği
hesaplıyor ve bunun tahmin hatasıyla ilişkisine bakıyoruz.

In [ ]:
morgan = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)


def parmak_izleri(smiles_listesi):
    izler = []
    for s in smiles_listesi:
        mol = Chem.MolFromSmiles(s)
        izler.append(morgan.GetFingerprint(mol) if mol is not None else None)
    return izler


print("Parmak izleri hesaplaniyor...")
iz_egitim = [f for f in parmak_izleri(smiles.iloc[egitim_idx]) if f is not None]
iz_test   = parmak_izleri(smiles.iloc[test_idx])

print("Egitim setine en yuksek benzerlik hesaplaniyor...")
en_yuksek_benzerlik = []
for iz in iz_test:
    if iz is None:
        en_yuksek_benzerlik.append(np.nan)
        continue
    benzerlikler = DataStructs.BulkTanimotoSimilarity(iz, iz_egitim)
    en_yuksek_benzerlik.append(max(benzerlikler))

df_ad["en_yuksek_tanimoto"] = en_yuksek_benzerlik

# Tanimoto kolonu eklendigi icin AD analiz dosyasini guncelleyerek
# yeniden kaydediyoruz (Bolum 7'de kolon henuz yoktu).
df_ad.to_csv(PROJE / "ntrk1_ad_test_analizi.csv", index=False)

print("\nTest molekullerinin egitim setine benzerligi")
print("-" * 58)
print(df_ad["en_yuksek_tanimoto"].describe().round(4).to_string())

In [ ]:
df_ad["benzerlik_dilim"] = pd.qcut(df_ad["en_yuksek_tanimoto"], 5,
                                   labels=["1 (en benzemez)", "2", "3", "4",
                                           "5 (en benzer)"])

benzerlik_ozet = (df_ad.groupby("benzerlik_dilim", observed=True)
                  .agg(Molekul=("artik", "size"),
                       Ortalama_benzerlik=("en_yuksek_tanimoto", "mean"),
                       MAE=("artik", lambda s: s.abs().mean()),
                       RMSE=("artik", lambda s: float(np.sqrt((s ** 2).mean())))))

print("Benzerlik dilimlerine gore tahmin hatasi")
print("=" * 78)
print(benzerlik_ozet.round(4).to_string())

fig, (sol, sag) = plt.subplots(1, 2, figsize=(13, 4.4))

sol.bar(range(len(benzerlik_ozet)), benzerlik_ozet["RMSE"], color="#8e44ad",
        edgecolor="white")
sol.set_xticks(range(len(benzerlik_ozet)))
sol.set_xticklabels(benzerlik_ozet.index, rotation=20, fontsize=8)
sol.set_ylabel(f"RMSE ({HEDEF_KOLON})")
sol.set_xlabel("Egitim setine benzerlik dilimi")
sol.set_title("Benzerlik dustukce hata artiyor mu?")
sol.grid(alpha=0.3, axis="y", linestyle="--")

sag.scatter(df_ad["en_yuksek_tanimoto"], df_ad["artik"].abs(), s=14,
            alpha=0.5, color="#8e44ad")
sag.set_xlabel("Egitim setine en yuksek Tanimoto benzerligi")
sag.set_ylabel(f"|hata| ({HEDEF_KOLON})")
sag.set_title("Benzerlik ile mutlak hata")
sag.grid(alpha=0.3, linestyle="--")

plt.tight_layout()
plt.savefig(PROJE / "ntrk1_ad_tanimoto.png", dpi=150, bbox_inches="tight")
plt.show()

r_benzerlik = df_ad["en_yuksek_tanimoto"].corr(df_ad["artik"].abs())
en_az = benzerlik_ozet["RMSE"].iloc[0]
en_cok = benzerlik_ozet["RMSE"].iloc[-1]
print(f"\nBenzerlik ile |hata| korelasyonu: r = {r_benzerlik:.3f}")
print(f"  (negatif olmasi beklenir: benzerlik arttikca hata azalir)")
print(f"En benzemez dilim RMSE : {en_az:.4f}")
print(f"En benzer dilim RMSE   : {en_cok:.4f}")

---
## 9. AD parametrelerini kaydetmek

Oturum 6'da yeni moleküller için tahmin üretirken bu sınırları kullanacağız.
Her tahminin yanında "bu molekül uygulanabilirlik alanı içinde mi?" bilgisini
vereceğiz.

### Benzerlik eşiği olarak ne seçmeli? — öğretici bir tuzak

Akla gelen ilk yöntem şudur: *test moleküllerinin benzerlik dağılımının alt
yüzdebirliğini eşik al.* Bu, kurs hazırlığı sırasında denendi ve **işe
yaramadı**. Nedeni öğreticidir.

Test setimiz eğitim setiyle **aynı kimyasal uzaydan** gelir: ikisi de aynı
kinaz inhibitörü serilerinden rastgele bölünmüştür. Bu yüzden test
moleküllerinin eğitim setine benzerliği çok yüksektir — 5. yüzdebirlik bile
**0,60 civarındadır**. Bu değeri eşik yaparsanız, farklı bir kimyasal uzaydan
gelen bir kütüphaneyi (Oturum 6'daki onaylanmış ilaçlar gibi) taradığınızda
moleküllerin neredeyse tamamı "alan dışı" damgası yer ve elinizde hiç aday
kalmaz.

**Doğru yaklaşım:** eşiği, taranacak kütüphaneye değil, **modelin hata
davranışına** bakarak seçmek. Yukarıdaki benzerlik dilimleri tablosu bunun
kanıtını verir: hata benzerlik düştükçe artar, ama en benzemez dilimde bile
kullanılamaz hâle gelmez.

Bu nedenle kemoinformatikte yaygın kabul gören **Tanimoto = 0,30** sınırını
kullanıyoruz (ECFP4 ile kabaca "ilgili kimyasal sınıf" sınırı). Aşağıda hem bu
değeri hem de veriden gelen yüzdebirliği hesaplayıp künyeye kaydediyoruz;
böylece ikisini karşılaştırabilirsiniz.

> **Genel ders.** Uygulanabilirlik alanı eşiği, **modelin hangi hata düzeyine
> kadar kabul edilebilir sayıldığına** dair bir karardır. Veriden otomatik
> türetilen her sayı, her bağlamda anlamlı olmaz.

In [ ]:
# Veriden gelen deger (yalnizca bilgi amacli): test setinin 5. yuzdebirligi
veri_esigi = float(np.nanpercentile(df_ad["en_yuksek_tanimoto"], 5))

# Kullanilacak esik: kemoinformatikte yaygin kabul goren sinir
benzerlik_esigi = TANIMOTO_ESIGI

print("Tanimoto esigi secimi")
print("-" * 60)
print(f"  Test setinin 5. yuzdebirligi : {veri_esigi:.3f}"
      "   (ayni kimyasal uzay -> asiri katı)")
print(f"  KULLANILAN esik              : {benzerlik_esigi:.3f}"
      "   (yaygin ECFP4 siniri)")
print(f"  Test setinde esigin altinda kalan: "
      f"{int((df_ad['en_yuksek_tanimoto'] < benzerlik_esigi).sum())} / {len(df_ad)}")
print()

ad_parametreleri = {
    "model_adi": paket_reg["model_adi"],
    "ozellik_seti": paket_reg["ozellik_seti"],
    "kaldirac": {
        "h_kritik": float(h_kritik),
        "egitim_n": int(n_egitim),
        "ozellik_p": int(p_ozellik),
        "formul": "3*(p+1)/n",
        "olcekleyici_ortalama": olcekleyici_ad.mean_.tolist(),
        "olcekleyici_std": olcekleyici_ad.scale_.tolist(),
    },
    "artik": {
        "sigma": float(sigma),
        "esik_sigma": 3.0,
    },
    "tanimoto": {
        "esik": benzerlik_esigi,
        "test_5_yuzdebirlik": veri_esigi,
        "parmak_izi": "Morgan radius=2, 2048 bit",
        "aciklama": "Egitim setine en yuksek Tanimoto benzerligi bu esigin "
                    "altindaysa molekul AD disinda kabul edilir.",
    },
    "test_ad_dagilimi": ozet_ad["Molekul"].to_dict(),
    "kaldirac_hata_korelasyonu": float(korelasyon),
    "tanimoto_hata_korelasyonu": float(r_benzerlik),
}

yol_ad = PROJE / "ntrk1_ad_parametreleri.json"
yol_ad.write_text(json.dumps(ad_parametreleri, ensure_ascii=False, indent=2),
                  encoding="utf-8")

# Egitim seti SMILES'larini da kaydet (Oturum 6'da Tanimoto icin gerekli)
pd.DataFrame({"Smiles": smiles.iloc[egitim_idx].values,
              HEDEF_KOLON: y_egitim.values}).to_csv(
    PROJE / "ntrk1_egitim_seti_smiles.csv", index=False)

print("Kaydedilen AD parametreleri")
print("-" * 60)
print(f"  Kritik kaldirac h*        : {h_kritik:.4f}")
print(f"  Artik sigma               : {sigma:.4f}")
print(f"  Tanimoto esigi            : {benzerlik_esigi:.4f}")
print(f"\n  {yol_ad.name}")
print(f"  ntrk1_egitim_seti_smiles.csv")

---
## 10. Özet

### Ne yaptık?

1. Oturum 3 ve 4'ün bütün sonuçlarını bir araya getirip **model ailelerini**
   ve **temsilleri** karşılaştırdık.
2. **SHAP** ile modeli hem küresel (hangi özellikler önemli) hem yerel
   (bu molekül neden potent) düzeyde açıkladık.
3. SHAP'ın **toplamsallık** özelliğini sayısal olarak doğruladık.
4. **Williams grafiği** ile uygulanabilirlik alanını kurduk: kaldıraç ve
   standartlaştırılmış artıklar.
5. AD'nin **gerçekten işe yarayıp yaramadığını** sınadık: uzak moleküllerde
   hata daha yüksek mi?
6. Alternatif olarak **Tanimoto benzerliği** tabanlı AD kurduk.
7. Bütün AD parametrelerini Oturum 6 için kaydettik.

### Akılda kalması gerekenler

- **SHAP, modelin ne gördüğünü söyler; doğanın ne yaptığını söylemez.**
  Nedensellik çıkarmak için kontrollü deney gerekir.
- **Korelasyonlu özellikler SHAP'ı böler.** Oturum 2'deki korelasyon filtresi
  bu yüzden yorumlanabilirlik için de gereklidir.
- **Her tahminin bir güven derecesi olmalı.** AD, modelin "bilmiyorum"
  demesinin yoludur. OECD QSAR ilkelerinde zorunludur.
- **AD'yi sınamadan kullanmayın.** Uzak moleküllerde hata artmıyorsa, o ölçüt
  o veri için işe yaramıyor demektir.
- **Tanimoto tabanlı AD, kimyacıyla konuşurken kaldıraçtan daha anlaşılırdır.**

### Sırada ne var?

**Oturum 6 — Yeni moleküllerin taranması ve SDF çıktısı.** Son oturumda
eğittiğimiz iki modeli birlikte kullanarak onaylanmış ilaç kütüphanesini
tarayacak, AD kontrolünden geçen adayları sıralayacak ve sonuçları
**SDF dosyaları** olarak (tek birleşik dosya + molekül başına ayrı dosya)
Google Drive'a kaydedeceğiz.

---

### Alıştırmalar

1. SHAP analizini **parmak izi** tabanlı bir modelle yapın. En önemli bitleri
   bulup `rdFingerprintGenerator` ile hangi alt-yapılara karşılık geldiklerini
   çözün. (İpucu: `GetMorganGenerator` ile `additionalOutput` kullanılır.)
2. `SHAP_ORNEK`'i 200 ve 1500 yapıp küresel önem sıralamasının ne kadar
   değiştiğine bakın. Kaç molekül yeterli?
3. Williams grafiğinde AD dışında kalan molekülleri çizin. Ortak bir yapısal
   özellikleri var mı?
4. Tanimoto eşiğini 0.3 ve 0.5 olarak sabitleyin. Kaç test molekülü AD dışında
   kalıyor ve o moleküllerin hatası ne?
5. Sınıflandırma modeli için de Williams grafiği kurulabilir mi? Artık yerine
   ne kullanırdınız? (İpucu: tahmin olasılığı ile gerçek etiket arasındaki fark)